In [6]:
import pandas as pd
pd.set_option('display.max_columns', None)   # показывать ВСЕ колонки
pd.set_option('display.width', None)         # не переносить строки
pd.set_option('display.max_colwidth', None)  # не обрезать содержимое ячеек


In [7]:
app_train_raw = pd.read_csv('../data/raw/application_train.csv')
app_train_columns_list = app_train_raw.columns.tolist()
description_data = pd.read_csv('../data/raw/HomeCredit_columns_description.csv', encoding='iso-8859-1')
trg_1 = (app_train_raw['TARGET'] == 1).sum()
trg_0 = (app_train_raw['TARGET'] == 0).sum()
print('Процент дефолтов: ',  trg_1*100/len(app_train_raw))
print('Процент ок: ',  trg_0*100/len(app_train_raw))
print('Всего записей:', len(app_train_raw))
print('Уникальных значений SK_ID_CURR:', app_train_raw['SK_ID_CURR'].nunique())
# description_data.head()

Процент дефолтов:  8.072881945686495
Процент ок:  91.92711805431351
Всего записей: 307511
Уникальных значений SK_ID_CURR: 307511


Target - поле показывающее, были ли у клиента проблемы с внесением платежей (просрочка)
Датасет train содержит tagret, в тесте target отсутствует. Наша задача на основе данных предсказать поведение клиента: \
TARGET = 1: просрочка платежа
TARGET = 0: все ок


Всего в таблице application находится 307511 записей с уникальными SK_ID_CURR (пропусков нет) \
Процент дефолта составляет 8,1% \
Процент ок: 91,9% \
Классы несбалансированны. Высокая доля правильных предсказаний не означает, что модель хорошо выявляет проблемные заявки.


In [17]:
def empty_data(df):
    cols = df.columns.tolist()
    vals = len(df)
    res = []
    for name in cols:
        counts = df[name].count()
        if counts < vals:
            res.append({ 'name': name, 'empty_count': vals - counts, 'empty_perc': round((vals - counts)*100/vals, 2) })
    res.sort(key=lambda x: x['empty_count'], reverse=True)
    return res

app_train_empties = pd.DataFrame(empty_data(app_train_raw))
print(app_train_empties.to_string())

                            name  empty_count  empty_perc
0                 COMMONAREA_AVG       214865       69.87
1                COMMONAREA_MODE       214865       69.87
2                COMMONAREA_MEDI       214865       69.87
3        NONLIVINGAPARTMENTS_AVG       213514       69.43
4       NONLIVINGAPARTMENTS_MODE       213514       69.43
5       NONLIVINGAPARTMENTS_MEDI       213514       69.43
6             FONDKAPREMONT_MODE       210295       68.39
7           LIVINGAPARTMENTS_AVG       210199       68.35
8          LIVINGAPARTMENTS_MODE       210199       68.35
9          LIVINGAPARTMENTS_MEDI       210199       68.35
10                 FLOORSMIN_AVG       208642       67.85
11                FLOORSMIN_MODE       208642       67.85
12                FLOORSMIN_MEDI       208642       67.85
13               YEARS_BUILD_AVG       204488       66.50
14              YEARS_BUILD_MODE       204488       66.50
15              YEARS_BUILD_MEDI       204488       66.50
16            

Среди столбцов с пропусками меньше всего пропусков DAYS_LAST_PHONE_CHANGE и CNT_FAM_MEMBERS \
Большая часть пропусков относится к данным о жилье и месте проживания клиента \

Из на первых взгляд существенных признаков много пропусков в OCCUPATION_TYPE (31%) \
Также интересны пропуски в OWN_CAR_AGE - скорее всего пропуски у тех клиентов, у которых машины нет - !!проверить \
также пропуски в числе запросов в bureau по поводу клиента, причем пропусков одинаковое количество, можно предположить, что у этих клиентов не было до этого кредитов и это их первый (!!проверить)